# DC-DC Converters

This notebook studies buck, boost, and buck-boost choppers in both continuous and discontinuous conduction modes using idealized switch and ripple waveforms.

## Theory Summary

Define the duty ratio `D`, switching period `T_s`, and the load parameter

$$K = \frac{2Lf_s}{R} = \frac{2L}{RT_s}.$$

### CCM conversion ratios

$$\text{Buck: } M(D)=\frac{V_o}{V_{in}}=D$$

$$\text{Boost: } M(D)=\frac{V_o}{V_{in}}=\frac{1}{1-D}$$

$$\text{Buck-boost: } M(D)=\frac{V_o}{V_{in}}=-\frac{D}{1-D}$$

Typical inductor ripple relations are

$$\Delta i_L^{buck}=\frac{(V_{in}-V_o)DT_s}{L}, \qquad \Delta i_L^{boost}=\frac{V_{in}DT_s}{L}.$$

### DCM conversion ratios

For the idealized models used in this library,

$$\text{Buck: } M=\frac{2}{1+\sqrt{1+4K/D^2}}$$

$$\text{Boost: } M=\frac{1}{2}\left(1+\sqrt{1+\frac{4D^2}{K}}\right)$$

$$\text{Buck-boost: } |M|=\frac{D}{\sqrt{K}}.$$

In DCM, the inductor current rises from zero, falls back to zero, and stays at zero for the remainder of the switching period.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from power_sim import ac_dc, dc_ac, dc_dc, loads, plotter, utils

plt.rcParams["figure.figsize"] = (11, 8)
plt.rcParams["axes.grid"] = True

In [ ]:
fs = 20_000.0
t = utils.timebase(fundamental_frequency=fs, cycles=6.0, samples_per_cycle=500)
tau = fs * t

buck_ccm = dc_dc.buck_converter(t, input_voltage=120.0, duty_cycle=0.4, switching_frequency=fs, resistance=10.0, inductance=2e-3, capacitance=220e-6, mode="CCM")
buck_dcm = dc_dc.buck_converter(t, input_voltage=120.0, duty_cycle=0.2, switching_frequency=fs, resistance=80.0, inductance=50e-6, capacitance=47e-6, mode="DCM")

fig, axes = plotter.plot_stacked_waveforms(
    tau,
    [
        {"y": buck_ccm["switch"], "label": "Gate signal", "ylabel": "s(t)", "color": "tab:blue"},
        {"y": buck_ccm["inductor_voltage"], "label": "Inductor voltage", "ylabel": r"$v_L$ (V)", "color": "tab:red"},
        {"y": buck_ccm["inductor_current"], "label": "Inductor current", "ylabel": r"$i_L$ (A)", "color": "tab:green"},
        {"y": buck_ccm["output_voltage"], "label": "Output voltage", "ylabel": r"$v_o$ (V)", "color": "tab:purple"},
    ],
    xlabel=r"$t/T_s$",
    title="Buck converter in CCM",
)
plt.show()

fig, axes = plotter.plot_stacked_waveforms(
    tau,
    [
        {"y": buck_dcm["switch"], "label": "Gate signal", "ylabel": "s(t)", "color": "tab:blue"},
        {"y": buck_dcm["inductor_current"], "label": "Inductor current", "ylabel": r"$i_L$ (A)", "color": "tab:orange"},
        {"y": buck_dcm["output_voltage"], "label": "Output voltage", "ylabel": r"$v_o$ (V)", "color": "tab:red"},
    ],
    xlabel=r"$t/T_s$",
    title="Buck converter in DCM",
)
plt.show()

print("Buck CCM average output voltage:", round(buck_ccm["average_output_voltage"], 3), "V")
print("Buck DCM average output voltage:", round(buck_dcm["average_output_voltage"], 3), "V")

## Extending to Other Loads

The chopper module produces ideal switch-level waveforms. If you want to study an `RL` or `RLE` load directly, pass the converter output voltage into `power_sim.loads` exactly as in the rectifier notebook.

In [ ]:
boost = dc_dc.boost_converter(t, input_voltage=60.0, duty_cycle=0.55, switching_frequency=fs, resistance=40.0, inductance=300e-6, capacitance=100e-6, mode="CCM")
buck_boost = dc_dc.buck_boost_converter(t, input_voltage=48.0, duty_cycle=0.45, switching_frequency=fs, resistance=30.0, inductance=250e-6, capacitance=100e-6, mode="DCM")

fig, axes = plotter.plot_stacked_waveforms(
    tau,
    [
        {"y": boost["inductor_current"], "label": "Boost inductor current", "ylabel": r"$i_L$ (A)", "color": "tab:blue"},
        {"y": boost["output_voltage"], "label": "Boost output", "ylabel": r"$v_o$ (V)", "color": "tab:red"},
        {"y": buck_boost["output_voltage"], "label": "Buck-boost output", "ylabel": r"$v_o$ (V)", "color": "tab:green"},
    ],
    xlabel=r"$t/T_s$",
    title="Boost and buck-boost examples",
)
plt.show()

print("Boost boundary inductance:", boost["boundary_inductance"], "H")
print("Buck-boost average output voltage:", round(buck_boost["average_output_voltage"], 3), "V")